# EXAMPLE (staff only): Mission 4 testing

This copy shows what a completed notebook looks like. The ideas are deliberately basic:
- **Steering:** raise the score of each required word's first token while generating, until that word appears (logit boosting).
- **Benchmark:** the share of 10 hand-written prompts whose story counts, using `common.evaluate_steering`.

Do not release this file to students.

### 1. Setup

In [1]:
# Get common.py (needed when running on a Colab kernel, including from VS Code)
!rm -f common.py
!wget -q https://raw.githubusercontent.com/sadhikamalladi/cs151b-fa26/project_testing/common.py
!head -1 common.py

"""CSE 151B Fall 2026 course project starter code:


In [2]:
!pip install -q transformers datasets

import os
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
os.makedirs("submission/benchmark_data", exist_ok=True)
print("Device:", common.get_device())

Device: cuda


### 2. Load the base model and practice prompts

In [3]:
model, tokenizer = common.load_model()

practice = common.make_word_prompts(100)
print("Example prompt:", practice[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  291MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Example prompt: {'prompt': 'Every morning,', 'words': ['truck', 'umbrella', 'sky']}


### 3. Your steering function

The template lists the words before the prompt, which scores about 0% because this small model does not follow instructions. This example changes how the model generates instead: at each step it adds `BOOST` to the score of the first token of every required word that has not appeared yet.

In [4]:
%%writefile submission/steer.py
import common
from transformers import LogitsProcessor, LogitsProcessorList

BOOST = 8.0  # how strongly to push the required words; higher = more words, less natural text


class BoostMissingWords(LogitsProcessor):
    """Add BOOST to the first token of each required word until that word appears."""

    def __init__(self, word_token_ids):
        self.word_token_ids = word_token_ids

    def __call__(self, input_ids, scores):
        for ids in self.word_token_ids:
            if not any((input_ids[0] == i).any() for i in ids):  # word not generated yet
                scores[:, ids] += BOOST
        return scores


def steer(model, tokenizer, prompt: str, words: list[str]) -> str:
    """Return a story continuing `prompt` that uses every word in `words`."""
    # First token of " word" and " Word", so the word can appear mid-sentence or at the start
    word_token_ids = [sorted({tokenizer(" " + w)["input_ids"][0], tokenizer(" " + w.capitalize())["input_ids"][0]})
                      for w in words]
    processors = LogitsProcessorList([BoostMissingWords(word_token_ids)])
    return common.generate_story(model, tokenizer, prompt, logits_processor=processors)

Writing submission/steer.py


In [5]:
steer = common.load_function("submission/steer.py", "steer")

result = common.evaluate_steering(steer, model, tokenizer, practice)
print(f"Word rate: {result['word_rate']:.0%}   Fluency (perplexity): {result['fluency_ppl']:.2f}")
print("Example:", practice[0]["words"], "→", result["stories"][0][:300])

Word rate: 31%   Fluency (perplexity): 2.06
Example: ['truck', 'umbrella', 'sky'] →  she would go to the sky and look up at the clouds. She would see different shapes and sizes, and she would imagine what the clouds could be like.

One day, she decided to try something new. She wanted to see if she could fly! She jumped up, and tried to take off, but she just couldn't do it. She ke


### 4. Your team benchmark

In [6]:
%%writefile submission/benchmark.py
"""Team benchmark: 10 hand-written prompts with words not in the practice word list."""
import common

PROMPTS = [
    {"prompt": "Once upon a time,", "words": ["owl", "bucket", "hill"]},
    {"prompt": "One day,", "words": ["candle", "puppy", "bed"]},
    {"prompt": "Lily and Ben", "words": ["sand", "shell", "wave"]},
    {"prompt": "Tom was", "words": ["coat", "button", "mom"]},
    {"prompt": "In a small town,", "words": ["baker", "bell", "street"]},
    {"prompt": "Every morning,", "words": ["chicken", "farm", "corn"]},
    {"prompt": "Sam had a", "words": ["drum", "song", "friend"]},
    {"prompt": "Mia wanted to", "words": ["boat", "lake", "fish"]},
    {"prompt": "There was a little", "words": ["turtle", "pond", "bug"]},
    {"prompt": "Long ago,", "words": ["king", "castle", "crown"]},
]


def team_benchmark(steer_fn, model, tokenizer) -> float:
    """Share of PROMPTS whose story counts (uses all words, 30+ words, fluent). Higher is better."""
    return common.evaluate_steering(steer_fn, model, tokenizer, PROMPTS)["word_rate"]

Writing submission/benchmark.py


In [7]:
team_benchmark = common.load_function("submission/benchmark.py", "team_benchmark")

print("Your steer function:")
common.check_team_benchmark(team_benchmark, model, tokenizer, steer_fn=steer)

Your steer function:
PASS: score 0.6000, runtime 12.2s (limit 300s)


0.6

### 5. Save your submission

In [8]:
common.package_submission("mission4")

Created /content/mission4_submission.zip (0 MB). Upload this file to Gradescope.


'/content/mission4_submission.zip'